In [3]:
import json
import pandas as pd
import scanpy as sc
from scipy.stats import spearmanr
from scipy.optimize import linear_sum_assignment

with open("benchmark_config.json") as f:
    config = json.load(f)

In [1]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

for ds in ["pbmc3k", "lung_65k"]:
    h100 = pd.read_csv(f"write/{ds}_rsc_gpu_015_harmonized_clusters.csv")
    l40s = pd.read_csv(f"write/{ds}_rsc_gpu_015_l40s_harmonized_clusters.csv")
    merged = h100.merge(l40s, on="barcode", suffixes=("_h100", "_l40s"))
    ari = adjusted_rand_score(merged["leiden_h100"], merged["leiden_l40s"])
    n_clusters_h100 = merged["leiden_h100"].nunique()
    n_clusters_l40s = merged["leiden_l40s"].nunique()
    identical = (merged["leiden_h100"] == merged["leiden_l40s"]).all()
    print(f"{ds}: H100 ({n_clusters_h100} cl) vs L40S ({n_clusters_l40s} cl) — ARI={ari:.6f} {'IDENTICAL' if identical else 'DIFFERENT'}")

pbmc3k: H100 (9 cl) vs L40S (9 cl) — ARI=1.000000 IDENTICAL
lung_65k: H100 (38 cl) vs L40S (34 cl) — ARI=0.934676 DIFFERENT


In [4]:
# Quick module score comparison
for ds, key in [("pbmc3k", "pbmc3k"), ("lung_65k", "lung65k")]:
    dcfg = config["datasets"][key]
    adata = sc.read_h5ad(dcfg["canonical_h5ad"])
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata, target_sum=10000)
    sc.pp.log1p(adata)
    
    h100 = pd.read_csv(f"write/{ds}_rsc_gpu_015_harmonized_clusters.csv")
    l40s = pd.read_csv(f"write/{ds}_rsc_gpu_015_l40s_harmonized_clusters.csv")
    
    for name, genes in dcfg["known_marker_sets"].items():
        present = [g for g in genes if g in adata.var_names]
        if len(present) >= 2:
            sc.tl.score_genes(adata, gene_list=present, score_name=f"score_{name}", use_raw=False)
    
    score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
    scores = adata.obs[score_cols].copy()
    scores.index = adata.obs_names.str.replace("-1$", "", regex=True)
    
    merged = h100.set_index("barcode").join(l40s.set_index("barcode"), lsuffix="_h100", rsuffix="_l40s")
    
    from scipy.stats import spearmanr
    from scipy.optimize import linear_sum_assignment
    
    matching = {}
    ct = pd.crosstab(merged["leiden_h100"], merged["leiden_l40s"])
    ri, ci = linear_sum_assignment(-ct.values)
    for i, j in zip(ri, ci):
        matching[ct.index[i]] = ct.columns[j]
    
    common = scores.index.intersection(merged.index)
    mean_h = scores.loc[common].assign(cl=merged.loc[common, "leiden_h100"]).groupby("cl").mean()
    mean_l = scores.loc[common].assign(cl=merged.loc[common, "leiden_l40s"]).groupby("cl").mean()
    
    rhos = []
    for ca, cb in matching.items():
        if ca in mean_h.index and cb in mean_l.index:
            r = spearmanr(mean_h.loc[ca], mean_l.loc[cb]).statistic
            rhos.append(r)
    
    avg_rho = sum(rhos) / len(rhos) if rhos else float('nan')
    print(f"{ds}: H100 vs L40S — Module rho = {avg_rho:.3f}")

pbmc3k: H100 vs L40S — Module rho = nan
lung_65k: H100 vs L40S — Module rho = 1.000


In [5]:
import json
import pandas as pd
import numpy as np
import scanpy as sc
from scipy.stats import spearmanr
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score

with open("benchmark_config.json") as f:
    config = json.load(f)

for ds, key in [("pbmc3k", "pbmc3k"), ("lung_65k", "lung65k")]:
    dcfg = config["datasets"][key]
    adata = sc.read_h5ad(dcfg["canonical_h5ad"])
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata, target_sum=10000)
    sc.pp.log1p(adata)
    
    for name, genes in dcfg["known_marker_sets"].items():
        present = [g for g in genes if g in adata.var_names]
        if len(present) >= 2:
            sc.tl.score_genes(adata, gene_list=present, score_name=f"score_{name}", use_raw=False)
    
    score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
    scores = adata.obs[score_cols].copy()
    scores.index = adata.obs_names.str.replace("-1$", "", regex=True)
    
    h100 = pd.read_csv(f"write/{ds}_rsc_gpu_015_harmonized_clusters.csv")
    l40s = pd.read_csv(f"write/{ds}_rsc_gpu_015_l40s_harmonized_clusters.csv")
    merged = h100.set_index("barcode").join(l40s.set_index("barcode"), lsuffix="_h100", rsuffix="_l40s")
    
    ct = pd.crosstab(merged["leiden_h100"], merged["leiden_l40s"])
    ri, ci = linear_sum_assignment(-ct.values)
    matching = {ct.index[i]: ct.columns[j] for i, j in zip(ri, ci)}
    
    common = scores.index.intersection(merged.index)
    mean_h = scores.loc[common].assign(cl=merged.loc[common, "leiden_h100"]).groupby("cl").mean()
    mean_l = scores.loc[common].assign(cl=merged.loc[common, "leiden_l40s"]).groupby("cl").mean()
    
    rhos = []
    for ca, cb in matching.items():
        if ca in mean_h.index and cb in mean_l.index:
            r = spearmanr(mean_h.loc[ca], mean_l.loc[cb]).statistic
            rhos.append(r)
    
    ari = adjusted_rand_score(merged["leiden_h100"], merged["leiden_l40s"])
    avg_rho = np.mean(rhos) if rhos else float('nan')
    print(f"{ds}: H100 vs L40S — ARI={ari:.4f}  Module rho={avg_rho:.3f}  ({len(matching)} matched clusters)")
    del adata

pbmc3k: H100 vs L40S — ARI=1.0000  Module rho=nan  (9 matched clusters)
lung_65k: H100 vs L40S — ARI=0.9347  Module rho=1.000  (34 matched clusters)


In [9]:
import json, time
import numpy as np
import pandas as pd
import scanpy as sc
import rapids_singlecell as rsc
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import KNeighborsTransformer
from scipy.stats import spearmanr
from scipy.optimize import linear_sum_assignment
import cupy as cp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
rmm.reinitialize(pool_allocator=True, managed_memory=True)
cp.cuda.set_allocator(rmm_cupy_allocator)

with open("benchmark_config.json") as f:
    config = json.load(f)
gcfg = config["global"]
dcfg = config["datasets"]["pbmc3k"]
prefix = dcfg["pipeline_prefix"]

SEEDS = [0, 1, 42]
cluster_results = {}

for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    t0 = time.time()
    
    adata = sc.read_h5ad(dcfg["canonical_h5ad"])
    adata.X = adata.layers["counts"].copy()
    
    # HVG on CPU counts layer first (same as your pipeline scripts)
    sc.pp.highly_variable_genes(adata, layer="counts", n_top_genes=dcfg["n_top_genes"],
                                flavor="seurat_v3", subset=False, inplace=True)
    
    # Now move to GPU
    rsc.get.anndata_to_GPU(adata)
    rsc.pp.normalize_total(adata, target_sum=gcfg["target_sum"])
    rsc.pp.log1p(adata)
    rsc.pp.pca(adata, n_comps=gcfg["pca_n_comps"], mask_var="highly_variable")
    rsc.pp.neighbors(adata, n_neighbors=dcfg["n_neighbors"], n_pcs=dcfg["neighbors_n_pcs"],
                     use_rep="X_pca", algorithm="brute", metric=gcfg["neighbor_metric"],
                     method=gcfg["neighbor_method"], random_state=seed)
    rsc.tl.leiden(adata, resolution=dcfg["leiden_resolution"], random_state=seed, key_added="leiden")
    rsc.get.anndata_to_CPU(adata)
    
    clusters = pd.Series(adata.obs["leiden"].astype(str).values,
                         index=adata.obs_names.str.replace("-1$", "", regex=True))
    cluster_results[seed] = clusters
    
    n_cl = clusters.nunique()
    print(f"  {n_cl} clusters, {time.time()-t0:.1f}s")
    
    # Save
    pd.DataFrame({"barcode": adata.obs_names.astype(str),
                   "leiden": adata.obs["leiden"].astype(str).values}
    ).to_csv(f"write/{prefix}_rsc_gpu_015_seed{seed}_clusters.csv", index=False)
    del adata

# Compare all pairs
print(f"\n{'='*50}")
print("Seed robustness: pairwise ARI")
print(f"{'='*50}")
from itertools import combinations
for sa, sb in combinations(SEEDS, 2):
    ca, cb = cluster_results[sa], cluster_results[sb]
    common = ca.index.intersection(cb.index)
    ari = adjusted_rand_score(ca.loc[common], cb.loc[common])
    identical = (ca.loc[common] == cb.loc[common]).all()
    print(f"  Seed {sa} vs {sb}: ARI={ari:.6f}  {'IDENTICAL' if identical else 'DIFFERENT'}")

# Variance summary
aris = []
for sa, sb in combinations(SEEDS, 2):
    common = cluster_results[sa].index.intersection(cluster_results[sb].index)
    aris.append(adjusted_rand_score(cluster_results[sa].loc[common], cluster_results[sb].loc[common]))
print(f"\n  Mean ARI: {np.mean(aris):.4f}")
print(f"  Std ARI:  {np.std(aris):.4f}")
print(f"  Min ARI:  {np.min(aris):.4f}")


--- Seed 0 ---
  8 clusters, 3.0s

--- Seed 1 ---
  8 clusters, 0.3s

--- Seed 42 ---
  9 clusters, 0.3s

Seed robustness: pairwise ARI
  Seed 0 vs 1: ARI=0.871538  DIFFERENT
  Seed 0 vs 42: ARI=0.947549  DIFFERENT
  Seed 1 vs 42: ARI=0.846176  DIFFERENT

  Mean ARI: 0.8884
  Std ARI:  0.0431
  Min ARI:  0.8462
